In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json, pickle, math, pytz, warnings
import numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, date, time, timedelta

warnings.filterwarnings("ignore")

# Paths
BASE_DIR = Path("/content/drive/MyDrive/xdte_selector/backtest_data")
KIT_DIR  = BASE_DIR / "_analysis_final_strategy" / "live_kit"
OUT_DIR  = BASE_DIR / "_daily_runs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

BOOKS = ["PUTS_0DTE_11","CALLS_0DTE_11","PUTS_1DTE_1515","CALLS_1DTE_1515"]
TZ_NY = pytz.timezone("America/New_York")

# Load live kit (frozen)
kit = json.load(open(KIT_DIR / "live_kit.json"))
features_by_book = kit["meta"]["features_by_book"]
hybrid_policy    = kit["hybrid_policy"]

def load_model(book):
    with open(KIT_DIR / f"{book}_model.pkl","rb") as f:
        return pickle.load(f)

def pf(s):
    s = pd.Series(s).dropna(); g=s[s>0].sum(); l=-s[s<0].sum()
    return float(g/l) if l>0 else (math.inf if g>0 else np.nan)

def cvar(s, alpha=0.05):
    v = pd.Series(s).dropna().values
    if v.size==0: return np.nan
    k=max(1,int(np.floor(alpha*len(v)))); idx=np.argsort(v)
    return float(np.mean(v[idx[:k]]))

In [ ]:
import yfinance as yf

def last_trading_day_for_intraday(ticker="^VIX", interval="1m", lookback_days=7):
    """Find most-recent day with intraday bars within lookback window."""
    for d in range(lookback_days):
        day_dt = (datetime.now(TZ_NY) - timedelta(days=d)).date()
        df = yf.Ticker(ticker).history(period="1d", interval=interval, auto_adjust=False)
        if not df.empty:
            idx = df.tz_convert(TZ_NY) if df.index.tz is not None else df.tz_localize(TZ_NY)
            if (idx.index.date==day_dt).any():
                return day_dt
    return datetime.now(TZ_NY).date()

def get_intraday_series(ticker, day_dt, interval="1m"):
    df = yf.Ticker(ticker).history(period="1d", interval=interval, auto_adjust=False)
    if df.empty: return pd.DataFrame()
    df = df.tz_convert(TZ_NY) if df.index.tz is not None else df.tz_localize(TZ_NY)
    return df[df.index.date==day_dt]

def last_close_series(ticker, days=400):
    df = yf.Ticker(ticker).history(period=f"{days}d", interval="1d", auto_adjust=False)
    if df.empty: return pd.DataFrame()
    df = df.tz_convert(TZ_NY) if df.index.tz is not None else df.tz_localize(TZ_NY)
    df["open_date"] = df.index.date
    return df

# Decide today's trade day automatically (most recent day with intraday bars)
TODAY_NY = last_trading_day_for_intraday("^VIX", "1m", lookback_days=7)

# Daily series for lags
spx_d = last_close_series("^GSPC")
vix_d = last_close_series("^VIX")
vix3_d= last_close_series("^VIX3M")
vvix_d= last_close_series("^VVIX")

# Lag day = most recent shared daily point before/at TODAY_NY
common_days = sorted(set(spx_d["open_date"]) & set(vix_d["open_date"]))
lag_day = common_days[-1]

# Build lag features
spx_row = spx_d[spx_d["open_date"]==lag_day].iloc[0]
vix_row = vix_d [vix_d ["open_date"]==lag_day].iloc[0]
vix3_row= vix3_d[vix3_d["open_date"]==lag_day].iloc[0] if not vix3_d.empty else None

L1_VIX_Close = float(vix_row["Close"])
ts_eff = float(L1_VIX_Close / vix3_row["Close"]) if vix3_row is not None and vix3_row["Close"]>0 else np.nan

spx_d["range"] = spx_d["High"] - spx_d["Low"]
spx_d["atr14"] = spx_d["range"].ewm(span=14, adjust=False).mean()
L1_SPX_ATR_Pct = float(spx_d.loc[spx_d["open_date"]==lag_day,"atr14"].iloc[0] / spx_row["Close"])

spx_d["rollmax"] = spx_d["Close"].rolling(252, min_periods=20).max()
rollmax = float(spx_d.loc[spx_d["open_date"]==lag_day,"rollmax"].iloc[0])
L1_SPX_Drawdown_Pct = (spx_row["Close"] - rollmax)/rollmax if rollmax>0 else np.nan

spx_d["ret"] = spx_d["Close"].pct_change()
rv5  = float(spx_d["ret"].rolling(5).std().iloc[-1]*(252**0.5))
rv20 = float(spx_d["ret"].rolling(20).std().iloc[-1]*(252**0.5))
L1_vix_pct = float(vix_d["Close"].rank(pct=True).iloc[-1])

# VVIX kill flag (lagged)
if not vvix_d.empty:
    vvix_d["ema30"] = vvix_d["Close"].ewm(span=30, adjust=False).mean()
    vvix_last = float(vvix_d.loc[vvix_d["open_date"]==lag_day,"Close"].iloc[0])
    vvix_e30  = float(vvix_d.loc[vvix_d["open_date"]==lag_day,"ema30"].iloc[0])
    L1_vvix_above_ema30 = bool(vvix_last > vvix_e30)

    # NEW: Calculate L1_vvix_pct and L1_vvix_above_ema20
    vvix_d["ema20"] = vvix_d["Close"].ewm(span=20, adjust=False).mean()
    vvix_e20  = float(vvix_d.loc[vvix_d["open_date"]==lag_day,"ema20"].iloc[0])
    L1_vvix_above_ema20 = bool(vvix_last > vvix_e20)
    L1_vvix_pct = float(vvix_d["Close"].rank(pct=True).iloc[-1])
else:
    L1_vvix_above_ema30 = False
    # NEW: Initialize if vvix_d is empty
    L1_vvix_above_ema20 = False
    L1_vvix_pct = np.nan

# Intraday series for today
vix_i = get_intraday_series("^VIX", TODAY_NY, "1m")
spx_i = get_intraday_series("^GSPC", TODAY_NY, "1m")

def price_at_or_before(df, tdt):
    if df.empty: return np.nan
    df_ = df[df.index<=tdt]
    if df_.empty: return np.nan
    return float(df_["Close"].iloc[-1])

def today_open_price(df):
    if df.empty: return np.nan
    d0 = df[df.index.date==TODAY_NY]
    if d0.empty: return np.nan
    return float(d0["Open"].iloc[0])

SPX_open = today_open_price(spx_i)

def session_inputs(session_label):
    t = time(11,0) if session_label=="11:00" else time(15,15)
    tdt = datetime.combine(TODAY_NY, t).replace(tzinfo=TZ_NY)
    VIX_Entry = price_at_or_before(vix_i, tdt)
    SPX_entry = price_at_or_before(spx_i, tdt)
    move = (SPX_entry - SPX_open) if (not np.isnan(SPX_entry) and not np.isnan(SPX_open)) else np.nan
    return VIX_Entry, move


In [ ]:
# === LIVE DECISION ENGINE — BOTH TRADE WINDOWS AS OF NOW ===
# Assumes:
#   - kit = json.load(open(KIT_DIR / "live_kit.json"))
#   - features_by_book = kit["meta"]["features_by_book"]
#   - hybrid_policy = kit["hybrid_policy"]
#   - load_model(book) exists (loads pkl from KIT_DIR)
# Produces:
#   - 'decisions' DataFrame with one row per book (PUTS/CALLS x 0DTE/1DTE)
#   - prints compact lines with 'why'
#   - saves CSV/JSON under OUT_DIR

import yfinance as yf
import pytz, json, pickle, math
import numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, date, time, timedelta

TZ_NY = pytz.timezone("America/New_York")
NOW_NY = datetime.now(TZ_NY)

BOOKS = ["PUTS_0DTE_11","CALLS_0DTE_11","PUTS_1DTE_1515","CALLS_1DTE_1515"]
SESSION_BY_BOOK = {
    "PUTS_0DTE_11":   "11:00",
    "CALLS_0DTE_11":  "11:00",
    "PUTS_1DTE_1515": "15:15",
    "CALLS_1DTE_1515":"15:15",
}
SESSION_TO_TIME = {"11:00": time(11,0), "15:15": time(15,15)}

def get_intraday_series(ticker: str, day_dt: date, interval="1m") -> pd.DataFrame:
    df = yf.Ticker(ticker).history(period="5d", interval=interval, auto_adjust=False)
    if df.empty: return pd.DataFrame()
    df = df.tz_convert(TZ_NY) if df.index.tz is not None else df.tz_localize(TZ_NY)
    return df[df.index.date == day_dt]

def get_daily_series(ticker: str, days=400) -> pd.DataFrame:
    df = yf.Ticker(ticker).history(period=f"{days}d", interval="1d", auto_adjust=False)
    if df.empty: return pd.DataFrame()
    df = df.tz_convert(TZ_NY) if df.index.tz is not None else df.tz_localize(TZ_NY)
    df["open_date"] = df.index.date
    return df

def last_trade_day_with_minutes(ticker="^VIX", lookback_days=5) -> date:
    now = datetime.now(TZ_NY)
    for d in range(lookback_days):
        day_dt = (now - timedelta(days=d)).date()
        if not get_intraday_series(ticker, day_dt).empty:
            return day_dt
    return (now - timedelta(days=1)).date()

def price_at_or_before(df: pd.DataFrame, tdt: datetime) -> float:
    if df.empty: return np.nan
    sub = df[df.index <= tdt]
    return float(sub["Close"].iloc[-1]) if not sub.empty else np.nan

def day_open_price(df: pd.DataFrame, day_dt: date) -> float:
    if df.empty: return np.nan
    d0 = df[df.index.date == day_dt]
    return float(d0["Open"].iloc[0]) if not d0.empty else np.nan

def to_numeric(df):
    z = df.copy()
    for c in z.columns:
        if not (pd.api.types.is_numeric_dtype(z[c]) or pd.api.types.is_bool_dtype(z[c])):
            z[c] = pd.to_numeric(z[c], errors='coerce')
    return z

def score_book(book, feats, base_row):
    X = to_numeric(base_row[feats]).fillna(np.nan)
    m = load_model(book)
    return float(m.predict(X)[0])

# --- Resolve trade day from most recent intraday bars
TRADE_DAY = last_trade_day_with_minutes("^VIX", lookback_days=5)

# --- Daily lags & kill from TRADE_DAY-1 (same as training/live-kit)
spx_d  = get_daily_series("^GSPC")
vix_d  = get_daily_series("^VIX")
vix3_d = get_daily_series("^VIX3M")
vvix_d = get_daily_series("^VVIX")

common_days = sorted(set(spx_d["open_date"]) & set(vix_d["open_date"]))
if not common_days:
    raise RuntimeError("No overlapping daily data for SPX and VIX.")
LAG_DAY = common_days[-1]

L1_VIX_Close = float(vix_d.loc[vix_d["open_date"]==LAG_DAY, "Close"].iloc[0])
if not vix3_d.empty and (vix3_d["open_date"]==LAG_DAY).any():
    vix3_close = float(vix3_d.loc[vix3_d["open_date"]==LAG_DAY, "Close"].iloc[0])
    L1_TS = float(L1_VIX_Close / vix3_close) if vix3_close > 0 else np.nan
else:
    L1_TS = np.nan

spx_d["range"] = spx_d["High"] - spx_d["Low"]
spx_d["atr14"] = spx_d["range"].ewm(span=14, adjust=False).mean()
spx_close_L1 = float(spx_d.loc[spx_d["open_date"]==LAG_DAY, "Close"].iloc[0])
L1_SPX_ATR_Pct = float(spx_d.loc[spx_d["open_date"]==LAG_DAY, "atr14"].iloc[0] / spx_close_L1)
spx_d["rollmax"] = spx_d["Close"].rolling(252, min_periods=20).max()
rollmax = float(spx_d.loc[spx_d["open_date"]==LAG_DAY, "rollmax"].iloc[0])
L1_SPX_Drawdown_Pct = (spx_close_L1 - rollmax)/rollmax if rollmax>0 else np.nan

spx_d["ret"] = spx_d["Close"].pct_change()
L1_rv5  = float(spx_d["ret"].rolling(5).std().iloc[-1]*(252**0.5))
L1_rv20 = float(spx_d["ret"].rolling(20).std().iloc[-1]*(252**0.5))
L1_VIX_pct = float(vix_d["Close"].rank(pct=True).iloc[-1])

if not vvix_d.empty and (vvix_d["open_date"]==LAG_DAY).any():
    vvix_d["ema20"] = vvix_d["Close"].ewm(span=20, adjust=False).mean()
    vvix_d["ema30"] = vvix_d["Close"].ewm(span=30, adjust=False).mean()
    vvix_last = float(vvix_d.loc[vvix_d["open_date"]==LAG_DAY, "Close"].iloc[0])
    vvix_e20  = float(vvix_d.loc[vvix_d["open_date"]==LAG_DAY, "ema20"].iloc[0])
    vvix_e30  = float(vvix_d.loc[vvix_d["open_date"]==LAG_DAY, "ema30"].iloc[0])
    L1_vvix_above_ema20 = bool(vvix_last > vvix_e20)
    L1_vvix_above_ema30 = bool(vvix_last > vvix_e30)
    L1_vvix_pct = float(vvix_d["Close"].rank(pct=True).iloc[-1])
else:
    L1_vvix_above_ema20 = False
    L1_vvix_above_ema30 = False
    L1_vvix_pct = np.nan

# --- Intraday series for TRADE_DAY
spx_i = get_intraday_series("^GSPC", TRADE_DAY, "1m")
vix_i = get_intraday_series("^VIX",   TRADE_DAY, "1m")
SPX_open = day_open_price(spx_i, TRADE_DAY)

def base_row_for_session_as_of_now(session_label: str):
    """Use the latest intraday bar up to min(now, session_time)."""
    sess_time = SESSION_TO_TIME[session_label]
    sess_dt   = datetime.combine(TRADE_DAY, sess_time).replace(tzinfo=TZ_NY)
    eff_dt    = min(NOW_NY, sess_dt)  # <-- key: both windows 'as of now'
    VIX_Entry = price_at_or_before(vix_i, eff_dt)
    SPX_entry = price_at_or_before(spx_i, eff_dt)
    move = (SPX_entry - SPX_open) if (not np.isnan(SPX_entry) and not np.isnan(SPX_open)) else np.nan
    row = {
        "open_date": pd.Timestamp(TRADE_DAY),
        "effective_ts": eff_dt.isoformat(),
        "L1_TS": float(L1_TS),
        "L1_VIX_Close": float(L1_VIX_Close),
        "L1_VIX_pct": float(L1_VIX_pct),
        "L1_SPX_ATR_Pct": float(L1_SPX_ATR_Pct),
        "L1_SPX_Drawdown_Pct": float(L1_SPX_Drawdown_Pct),
        "L1_rv5": float(L1_rv5),
        "L1_rv20": float(L1_rv20),
        "DoW": pd.Timestamp(TRADE_DAY).weekday(),
        "L1_vvix_above_ema30": L1_vvix_above_ema30,
        "L1_vvix_above_ema20": L1_vvix_above_ema20,
        "L1_vvix_pct": L1_vvix_pct,
        "VIX_Entry_11": VIX_Entry if session_label=="11:00" else np.nan,
        "Intraday_Move_OpenToEntry_11": move if session_label=="11:00" else np.nan,
        "VIX_Entry_1515": VIX_Entry if session_label=="15:15" else np.nan,
        "Intraday_Move_OpenToEntry_1515": move if session_label=="15:15" else np.nan,
        "t0_VIX_change_from_close_11": (VIX_Entry - L1_VIX_Close) if session_label=="11:00" else np.nan,
        "t0_VIX_change_from_close_15": (VIX_Entry - L1_VIX_Close) if session_label=="15:15" else np.nan,
        "input_VIX_Entry": float(VIX_Entry) if not np.isnan(VIX_Entry) else None,
        "input_SPX_entry": float(SPX_entry) if not np.isnan(SPX_entry) else None,
        "input_SPX_open":  float(SPX_open)  if not np.isnan(SPX_open)  else None,
        "input_move":      float(move)      if not np.isnan(move)      else None,
    }
    return pd.DataFrame([row])

# --- Decisions per book: rule decision + hybrid override (as-of-now)
def decide_puts(book, score):
    p = kit["puts"][book]  # keep, tau_top, tau_bot, use_top
    tau = p["tau_top"] if p["use_top"] else p["tau_bot"]
    pass_gate = (score >= tau) if p["use_top"] else (score <= tau)
    action = "KEEP_SHORT" if pass_gate else "SKIP"
    margin = (score - tau) if p["use_top"] else (tau - score)
    why = f"score={round(score,3)} vs τ={round(tau,3)} ({'top' if p['use_top'] else 'bot'}-keep, margin={round(margin,1)})"
    return action, why, margin

def decide_calls(book, score):
    p = kit["calls"][book]
    edges = np.array(p["edges"])
    if len(np.unique(edges)) < 3 or np.isnan(score):
        return "SKIP", "no-bucket (edges<3 or NaN)", None
    dec = pd.cut([score], edges, labels=False, include_lowest=True)
    d = int(dec[0]) if not pd.isna(dec[0]) else None
    if d is None:
        return "SKIP", "no-bucket", None
    if d in p["long_deciles"]:
        why = f"decile={d} ∈ long_set"
        return "GO_LONG",  why, d
    if d in p["short_deciles"]:
        why = f"decile={d} ∈ short_set"
        return "GO_SHORT", why, d
    return "SKIP", f"decile={d} ∉ (long|short)", d

def hybrid_override(book, action, kill_flag, score):
    # tails on kill for calls (if configured), else gamma scale
    if not kill_flag:
        return action, 1.0, "kill=False"
    tails = hybrid_policy.get("tails", {}).get(book)
    if tails and book in kit.get("calls", {}):
        edges = np.array(kit["calls"][book]["edges"])
        if len(np.unique(edges)) >= 3 and not np.isnan(score):
            dec = pd.cut([score], edges, labels=False, include_lowest=True)
            n_bins = len(edges) - 1
            q_idx = (float(dec[0]) + 0.5)/n_bins if not pd.isna(dec[0]) else None
            ql, qh = tails
            if q_idx is not None:
                if q_idx <= ql: return "GO_LONG",  1.0, f"kill=True tail=low (≤{ql:.3f})"
                if q_idx >= qh: return "GO_SHORT", 1.0, f"kill=True tail=high (≥{qh:.3f})"
                return "SKIP", 1.0, "kill=True tail=mid"
    g = float(hybrid_policy.get("gammas", {}).get(book, 1.0))
    return action, g, f"kill=True gamma={g}"

rows=[]
for book in BOOKS:
    session = SESSION_BY_BOOK[book]
    base_row = base_row_for_session_as_of_now(session)
    kill_flag = bool(base_row["L1_vvix_above_ema30"].iloc[0])
    feats = features_by_book.get(book, [])
    score = score_book(book, feats, base_row)

    if book.startswith("PUTS"):
        rule_action, rule_why, _ = decide_puts(book, score)
    else:
        rule_action, rule_why, _ = decide_calls(book, score)

    final_action, gamma, hybrid_why = hybrid_override(book, rule_action, kill_flag, score)

    rows.append({
        "decision_ts": NOW_NY.isoformat(),
        "trade_day": str(TRADE_DAY),
        "session": session,
        "effective_ts": base_row["effective_ts"].iloc[0],  # the bar we used
        "book": book,
        "kill_day": kill_flag,
        "score": round(score,3),
        "rule_action": rule_action,
        "final_action": final_action,
        "size_gamma": gamma,
        "why": f"{rule_why}; {hybrid_why}",
        "input_VIX_Entry": base_row["input_VIX_Entry"].iloc[0],
        "input_SPX_entry": base_row["input_SPX_entry"].iloc[0],
        "input_SPX_open":  base_row["input_SPX_open"].iloc[0],
        "input_move":      base_row["input_move"].iloc[0],
    })

decisions = pd.DataFrame(rows)

# Print compact lines
def fmt_row(r):
    a = r["final_action"]; g = r.get("size_gamma",1.0)
    return (f"{r['session']}  {r['book']}: {a}  gamma={g}  "
            f"| as_of={r['effective_ts']}  | why: {r['why']}")

print("\n=== LIVE DECISIONS (both sessions, as-of-now) ===")
for _, r in decisions.iterrows():
    print(fmt_row(r))

# Persist run
#OUT_DIR = Path(OUT_DIR)
#OUT_DIR.mkdir(parents=True, exist_ok=True)
#stamp = datetime.now(TZ_NY).strftime("%Y%m%d_%H%M%S")
#csv_path  = OUT_DIR / f"decisions_asof_{stamp}.csv"
#json_path = OUT_DIR / f"decisions_asof_{stamp}.json"
#decisions.to_csv(csv_path, index=False)
#with open(json_path,"w") as f: json.dump(decisions.to_dict(orient="records"), f, indent=2)
#print(f"\n[saved] {csv_path.name}\n[saved] {json_path.name}")



=== LIVE DECISIONS (both sessions, as-of-now) ===
11:00  PUTS_0DTE_11: KEEP_SHORT  gamma=1.0  | as_of=2025-11-10T11:00:00-04:56  | why: score=-914.949 vs τ=-1520.909 (top-keep, margin=606.0); kill=False
11:00  CALLS_0DTE_11: GO_LONG  gamma=1.0  | as_of=2025-11-10T11:00:00-04:56  | why: decile=3 ∈ long_set; kill=False
15:15  PUTS_1DTE_1515: KEEP_SHORT  gamma=1.0  | as_of=2025-11-10T15:15:00-04:56  | why: score=-265.506 vs τ=-1206.29 (top-keep, margin=940.8); kill=False
15:15  CALLS_1DTE_1515: GO_SHORT  gamma=1.0  | as_of=2025-11-10T15:15:00-04:56  | why: decile=0 ∈ short_set; kill=False

[saved] decisions_asof_20251110_163534.csv
[saved] decisions_asof_20251110_163534.json
